## 8. Summary & Resources

### What We Covered
1. **Composition graphs**: Representing alloy compositions as graphs for GNN-based classification
2. **Glass-forming ability prediction**: Binary classification with a simple GCN
3. **Baseline comparison**: GNN vs. Random Forest on hand-crafted features
4. **Pretrained universal potentials**: Using M3GNet for metallic alloy energy prediction and structural relaxation
5. **Physics connection**: How crystal phase stability relates to glass-forming ability

### Key Takeaways
- GNNs can learn **element interactions** directly from data, even from composition alone
- For composition-only tasks, hand-crafted features (Magpie) can be competitive — GNNs shine when structure is available
- **Pretrained universal potentials** (M3GNet, CHGNet, MACE) are transformative for metallic alloy research
- Metallic glass prediction remains challenging — it sits at the intersection of thermodynamics, kinetics, and structure

### Datasets

| Dataset | Size | Type | Access |
|---------|------|------|--------|
| `matbench_glass` | ~5,000 | Binary GFA | `matminer.datasets.load_dataset` |
| `glass_ternary_landolt` | ~7,000 | Ternary glasses | `matminer.datasets.load_dataset` |
| Mendeley MG dataset | ~9,000 | BMG properties | https://data.mendeley.com/datasets/t4bf9fb9hd |
| Materials Project | ~150k+ | Crystal structures | https://materialsproject.org |

### Key Papers
1. Ren et al. "Accelerated discovery of metallic glasses through ML and high-throughput experiments." Science Advances 4, eaaq1566 (2018)
2. Ward et al. "A general-purpose machine learning framework for predicting properties of inorganic materials." npj Comp. Mat. 2, 16028 (2016)
3. Sun et al. "Inverse design of glass structure with deep graph neural networks." Nature Communications 12, 5359 (2021)
4. Chen & Ong. "A Universal Graph Deep Learning Interatomic Potential." Nature Comp. Sci. 2, 718 (2022)
5. Dunn et al. "Benchmarking Materials Property Prediction Methods." npj Comp. Mat. 6, 138 (2020)

### GitHub Repositories
- [`materialsvirtuallab/matgl`](https://github.com/materialsvirtuallab/matgl) — MEGNet & M3GNet (pretrained models)
- [`hackingmaterials/matminer`](https://github.com/hackingmaterials/matminer) — Materials data mining & featurization
- [`pyg-team/pytorch_geometric`](https://github.com/pyg-team/pytorch_geometric) — PyTorch Geometric for custom GNNs
- [`txie-93/cgcnn`](https://github.com/txie-93/cgcnn) — Crystal Graph CNN
- [`ACEsuit/mace`](https://github.com/ACEsuit/mace) — MACE equivariant potentials
- [`JuDFTteam/best-of-atomistic-machine-learning`](https://github.com/JuDFTteam/best-of-atomistic-machine-learning) — Curated list of atomistic ML projects

## 7. Discussion: Connecting Composition and Structure

### What We've Learned

**Part A** showed that even a simple GNN on composition graphs can classify glass formers — the graph structure captures element interactions that matter for GFA.

**Part B** demonstrated that pretrained universal potentials can predict energies and relax structures of metallic alloy phases — these energies directly relate to the thermodynamic driving force for crystallization vs. glass formation.

### The Full Picture

In a real research workflow, you might:
1. **Screen compositions** with the GNN classifier (Part A) → find promising glass-forming compositions
2. **Identify competing crystalline phases** from databases or structure prediction
3. **Calculate phase stability** using M3GNet (Part B) → estimate the thermodynamic barrier to crystallization
4. **Simulate glass formation** via melt-quench molecular dynamics with M3GNet → directly observe whether amorphization occurs

### Open Challenges
- **Amorphous structure datasets**: Unlike crystals, there's no "Materials Project for glasses" — generating training data requires expensive MD simulations
- **Long-range disorder**: Standard GNN cutoffs (~5Å) capture local order but may miss medium-range order important in glasses
- **Glass transition**: Predicting the glass transition temperature requires understanding dynamics, not just energy landscapes
- **Multi-component systems**: Real metallic glasses often have 5+ elements — the composition space is vast

In [ ]:
# Structural relaxation of CuZr — finding the ground state
from matgl.ext.ase import M3GNetCalculator
from pymatgen.io.ase import AseAtomsAdaptor
from ase.optimize import BFGS
import io

calc = M3GNetCalculator(potential=pot_model)

# Start with slightly distorted CuZr B2 structure
cuzr_distorted = Structure(
    Lattice.cubic(3.35),  # slightly off
    ["Cu", "Zr"],
    [[0.01, 0.02, 0.0], [0.49, 0.51, 0.50]]
)

atoms = AseAtomsAdaptor.get_atoms(cuzr_distorted)
atoms.calc = calc

e_before = atoms.get_potential_energy()
f_before = np.max(np.abs(atoms.get_forces()))

# Relax
optimizer = BFGS(atoms, logfile=io.StringIO())
optimizer.run(fmax=0.05, steps=50)

e_after = atoms.get_potential_energy()
f_after = np.max(np.abs(atoms.get_forces()))
relaxed = AseAtomsAdaptor.get_structure(atoms)

print("CuZr B2 Structure Relaxation:")
print(f"  Before: E = {e_before:.4f} eV, max|F| = {f_before:.4f} eV/Å, a = {cuzr_distorted.lattice.a:.3f} Å")
print(f"  After:  E = {e_after:.4f} eV, max|F| = {f_after:.4f} eV/Å, a = {relaxed.lattice.a:.3f} Å")
print(f"  Energy change: {e_after - e_before:.4f} eV")
print(f"\nThis tells us how stable the crystalline CuZr phase is.")
print("Deeply negative formation energy = strong crystalline competitor = harder to form glass.")

In [ ]:
import matgl
from pymatgen.core import Structure, Lattice

# Load the pretrained M3GNet universal potential
pot_model = matgl.load_model("M3GNet-MP-2021.2.8-PES")
print("Loaded M3GNet Universal Potential")

# Create crystal structures for common metallic glass-forming systems
# These are the CRYSTALLINE phases that compete with glass formation

metallic_structures = {
    # Cu-Zr system — the most studied metallic glass system
    "Cu (FCC)": Structure(
        Lattice.cubic(3.61), ["Cu"], [[0, 0, 0]]
    ),
    "Zr (HCP)": Structure(
        Lattice.hexagonal(3.23, 5.15), ["Zr", "Zr"], [[1/3, 2/3, 1/4], [2/3, 1/3, 3/4]]
    ),
    "CuZr (B2)": Structure(
        Lattice.cubic(3.26), ["Cu", "Zr"], [[0, 0, 0], [0.5, 0.5, 0.5]]
    ),
    "Cu2Zr": Structure(
        Lattice.cubic(6.85), 
        ["Cu", "Cu", "Cu", "Cu", "Cu", "Cu", "Cu", "Cu", "Zr", "Zr", "Zr", "Zr"],
        [[0.25,0.25,0.25], [0.25,0.75,0.75], [0.75,0.25,0.75], [0.75,0.75,0.25],
         [0.75,0.75,0.75], [0.75,0.25,0.25], [0.25,0.75,0.25], [0.25,0.25,0.75],
         [0,0,0], [0,0.5,0.5], [0.5,0,0.5], [0.5,0.5,0]]
    ),
    # Ni-Nb system — another good glass former
    "Ni (FCC)": Structure(
        Lattice.cubic(3.52), ["Ni"], [[0, 0, 0]]
    ),
    "Nb (BCC)": Structure(
        Lattice.cubic(3.30), ["Nb"], [[0, 0, 0]]
    ),
    # Ti-based
    "Ti (HCP)": Structure(
        Lattice.hexagonal(2.95, 4.68), ["Ti", "Ti"], [[1/3, 2/3, 1/4], [2/3, 1/3, 3/4]]
    ),
}

# Predict energies using M3GNet
print(f"\n{'Structure':<15} {'Energy/atom (eV)':>18} {'Natoms':>8}")
print("-" * 45)
for name, struct in metallic_structures.items():
    energy = float(pot_model.predict_structure(struct))
    n_atoms = len(struct)
    print(f"{name:<15} {energy/n_atoms:>18.4f} {n_atoms:>8}")

---
## Part B: Structure-Based Approach with Pretrained Universal Potentials

While Part A worked purely from composition, the real power of GNNs comes from learning on **3D atomic structures**. Universal interatomic potentials like **M3GNet** were trained on the entire Materials Project and can predict energies for any atomic configuration — including metallic alloy structures relevant to glass formers.

### Why This Matters for Metallic Glasses:
- Crystal structures of competing phases tell us about the **energy landscape** 
- The more stable the crystalline phases, the harder it is to form a glass
- M3GNet can also be used for molecular dynamics to simulate the glass formation process

In [ ]:
# Random Forest baseline using matminer composition features
from matminer.featurizers.composition import ElementProperty
from sklearn.ensemble import RandomForestClassifier

# Reload fresh copy of dataset for RF
df_rf = load_dataset("matbench_glass")
df_rf["composition"] = df_rf["composition"].apply(
    lambda x: Composition(x) if isinstance(x, str) else x
)

# Generate Magpie composition features
print("Generating composition features (this may take a minute)...")
ep = ElementProperty.from_preset("magpie")
df_rf = ep.featurize_dataframe(df_rf, "composition", ignore_errors=True)

# Prepare feature matrix
feature_cols = [c for c in df_rf.columns if c not in ["composition", "gfa"]]
df_rf = df_rf.dropna(subset=feature_cols)

X = df_rf[feature_cols].values
y = df_rf["gfa"].values.astype(float)

# Same split ratio
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_preds)
rf_auc = roc_auc_score(y_test, rf_probs)

print(f"\n=== Random Forest Results ===")
print(f"Accuracy: {rf_acc:.3f}")
print(f"ROC AUC:  {rf_auc:.3f}")
print(f"\n=== Comparison ===")
print(f"GNN Accuracy:  {gnn_acc:.3f} | RF Accuracy:  {rf_acc:.3f}")
print(f"GNN AUC:       {gnn_auc:.3f} | RF AUC:       {rf_auc:.3f}")
print(f"\nNote: The RF baseline uses 100+ hand-crafted Magpie features.")
print("The GNN uses only 6 raw elemental properties — yet captures element interactions!")

### 6. Baseline Comparison: Random Forest on Composition Features

A key question in any ML study: does the GNN actually add value over simpler methods? Let's train a Random Forest on hand-crafted composition features using `matminer`.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# GNN evaluation on test set
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in test_loader:
        out = model(batch.x, batch.edge_index, batch.batch)
        probs = torch.sigmoid(out)
        preds = (probs > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

gnn_acc = accuracy_score(all_labels, all_preds)
gnn_auc = roc_auc_score(all_labels, all_probs)

print("=== GNN Results ===")
print(f"Accuracy: {gnn_acc:.3f}")
print(f"ROC AUC:  {gnn_auc:.3f}")
print(f"\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=["Non-glass", "Glass"]))

### 5. Evaluate the GNN

Let's evaluate on the test set and compare with a Random Forest baseline.

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(train_losses, label="Train Loss", color='steelblue')
ax.plot(val_losses, label="Val Loss", color='coral')
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("Training & Validation Loss")
ax.legend()

ax = axes[1]
ax.plot(val_accs, label="Val Accuracy", color='forestgreen')
ax.axhline(y=max(val_accs), color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title(f"Validation Accuracy (best: {max(val_accs):.3f})")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Training loop
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = torch.nn.BCEWithLogitsLoss()

train_losses = []
val_losses = []
val_accs = []

for epoch in range(1, 51):
    # Train
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    train_loss = total_loss / len(train_graphs)
    train_losses.append(train_loss)
    
    # Validate
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch.x, batch.edge_index, batch.batch)
            val_loss += criterion(out, batch.y).item() * batch.num_graphs
            pred = (torch.sigmoid(out) > 0.5).float()
            correct += (pred == batch.y).sum().item()
            total += batch.num_graphs
    
    val_loss /= len(val_graphs)
    val_acc = correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f}")

print(f"\nBest validation accuracy: {max(val_accs):.3f} (epoch {np.argmax(val_accs)+1})")

In [ ]:
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F

class GlassGNN(torch.nn.Module):
    """Simple GNN for binary glass-forming ability classification.
    
    Architecture:
        Input → GCN → ReLU → GCN → ReLU → GlobalMeanPool → Linear → Sigmoid
    """
    def __init__(self, in_channels, hidden_channels=32):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 1)
    
    def forward(self, x, edge_index, batch):
        # Message passing
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        
        # Aggregate node features to graph level
        x = global_mean_pool(x, batch)
        
        # Classify
        return self.lin(x).squeeze(-1)

# Initialize model
in_features = 6  # [Z, electronegativity, radius, row, group, fraction]
model = GlassGNN(in_channels=in_features, hidden_channels=32)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

### 4. Build and Train a GNN for Glass Classification

We build a simple Graph Neural Network using PyTorch Geometric:
- **2 GCN layers** for message passing between element nodes
- **Global mean pooling** to get a fixed-size representation of each composition
- **Linear classifier** for binary glass/non-glass prediction

This is intentionally simple to demonstrate the concept — more sophisticated architectures (GAT, GIN, etc.) could improve performance.

In [ ]:
# Convert all compositions to graphs
print("Converting compositions to graphs...")
graphs = []
skipped = 0
for idx, row in df.iterrows():
    try:
        g = composition_to_graph(row["composition"], row["gfa"])
        graphs.append(g)
    except Exception as e:
        skipped += 1

print(f"Successfully converted: {len(graphs)} graphs")
print(f"Skipped: {skipped}")

# Train/val/test split
from sklearn.model_selection import train_test_split

train_graphs, test_graphs = train_test_split(graphs, test_size=0.2, random_state=42)
train_graphs, val_graphs = train_test_split(train_graphs, test_size=0.15, random_state=42)

print(f"\nSplit: {len(train_graphs)} train / {len(val_graphs)} val / {len(test_graphs)} test")

# Create data loaders
train_loader = DataLoader(train_graphs, batch_size=64, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=64)
test_loader = DataLoader(test_graphs, batch_size=64)

In [ ]:
import torch
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from pymatgen.core import Element

def composition_to_graph(comp, label):
    """Convert a pymatgen Composition to a PyG graph.
    
    Nodes = elements, features = elemental properties + fraction
    Edges = fully connected (all pairs)
    """
    elements = list(comp.as_dict().keys())
    fractions = list(comp.as_dict().values())
    total = sum(fractions)
    fractions = [f / total for f in fractions]  # normalize
    
    # Node features: [atomic_number, electronegativity, atomic_radius, 
    #                  row, group, fraction]
    node_features = []
    for el, frac in zip(elements, fractions):
        elem = Element(el)
        node_features.append([
            elem.Z / 100.0,  # normalize atomic number
            elem.X if elem.X else 1.5,  # electronegativity (Pauling)
            float(elem.atomic_radius) if elem.atomic_radius else 1.5,
            elem.row / 7.0,
            elem.group / 18.0,
            frac,  # composition fraction
        ])
    
    x = torch.tensor(node_features, dtype=torch.float)
    
    # Fully connected edges (all elements interact)
    n = len(elements)
    if n > 1:
        edge_index = torch.tensor(
            [[i, j] for i in range(n) for j in range(n) if i != j],
            dtype=torch.long
        ).t().contiguous()
    else:
        # Single element — self-loop
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    
    y = torch.tensor([float(label)], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, y=y)

# Test with an example
test_comp = Composition("Cu50Zr50")
test_graph = composition_to_graph(test_comp, label=1)
print(f"Cu50Zr50 graph:")
print(f"  Nodes: {test_graph.num_nodes}, Edges: {test_graph.num_edges}")
print(f"  Node features shape: {test_graph.x.shape}")
print(f"  Node features (Cu): {test_graph.x[0].tolist()}")
print(f"  Node features (Zr): {test_graph.x[1].tolist()}")

### 3. Building Composition Graphs

Now comes the key idea: we represent each alloy composition as a graph that a GNN can process.

For a composition like Cu₅₀Zr₅₀:
- **Node 0**: Cu, with features [Z=29, χ=1.90, r=1.28Å, fraction=0.5]
- **Node 1**: Zr, with features [Z=40, χ=1.33, r=1.60Å, fraction=0.5]
- **Edge 0→1 and 1→0**: These elements interact in the alloy

In [ ]:
# Explore the dataset
from pymatgen.core import Composition

# Parse compositions
df["composition"] = df["composition"].apply(lambda x: Composition(x) if isinstance(x, str) else x)

# Extract some stats
df["n_elements"] = df["composition"].apply(lambda c: len(c.elements))
df["formula"] = df["composition"].apply(lambda c: c.reduced_formula)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Number of elements distribution
ax = axes[0]
df.groupby("gfa")["n_elements"].value_counts().unstack(0).plot.bar(ax=ax)
ax.set_xlabel("Number of Elements")
ax.set_ylabel("Count")
ax.set_title("Number of Elements by Glass Formation")
ax.legend(["Non-glass", "Glass"])
ax.tick_params(axis='x', rotation=0)

# Most common elements in glass formers
ax = axes[1]
glass_comps = df[df["gfa"] == True]["composition"]
element_counts = {}
for comp in glass_comps:
    for el in comp.elements:
        element_counts[str(el)] = element_counts.get(str(el), 0) + 1

top_elements = sorted(element_counts.items(), key=lambda x: x[1], reverse=True)[:15]
elements, counts = zip(*top_elements)
ax.barh(range(len(elements)), counts, color='steelblue')
ax.set_yticks(range(len(elements)))
ax.set_yticklabels(elements)
ax.set_xlabel("Frequency in Glass Formers")
ax.set_title("Most Common Elements in Metallic Glasses")
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
from matminer.datasets import load_dataset

# Load the metallic glass dataset
df = load_dataset("matbench_glass")
print(f"Dataset size: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nClass distribution:")
print(df["gfa"].value_counts())
print(f"\nSample entries:")
df.head(10)

### 2. Load the Metallic Glass Dataset

We use the `matbench_glass` dataset from matminer — a curated binary classification dataset where the task is to predict whether a given composition forms a metallic glass or not.

---
## Part A: Predicting Glass-Forming Ability from Composition

### The Glass-Forming Ability Problem

Not all alloy compositions can form metallic glasses. When a molten alloy is cooled rapidly (quenched), some compositions freeze into an amorphous solid while others crystallize. Predicting which compositions form glasses is a major open problem.

**Key factors affecting GFA:**
- Atomic size mismatch between elements (>12% helps)
- Negative heat of mixing (strong unlike-atom attraction)
- Multiple components (confusion principle — more elements = harder to crystallize)
- Deep eutectic compositions

### Our Approach: Composition as a Graph

We'll represent each alloy composition as a small graph:
- **Nodes** = elements present in the alloy
- **Node features** = elemental properties (atomic number, electronegativity, radius, etc.) + composition fraction
- **Edges** = fully connected (all elements interact with each other)

This lets us use GNNs even without 3D structure information!

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Imports successful!")

In [ ]:
# Install dependencies (uncomment if running in Colab)
# !pip install matminer pymatgen torch torch_geometric matgl matplotlib numpy pandas scikit-learn

## 1. Setup & Installation

We use the following libraries:
- **`matminer`** — Materials data mining: built-in datasets including metallic glass data
- **`pymatgen`** — Crystal structure manipulation and elemental properties
- **`torch` + `torch_geometric`** — PyTorch Geometric for building custom GNNs
- **`matgl`** — Pretrained universal potentials (M3GNet) for Part B
- **`scikit-learn`** — Baseline ML models for comparison

# GNNs for Metallic Glass & Amorphous Alloy Property Prediction

## AI4Physics Learning Workshop — Uppsala University, April 2026

**Tutorial:** Geometric Deep Learning: Hands-on

### What Are Metallic Glasses?

Metallic glasses (MGs) are **amorphous metals** — alloys that lack the long-range periodic order of crystals. Instead of arranging into a regular lattice, the atoms are frozen in a disordered configuration, similar to a liquid.

```
Crystal:          Metallic Glass:
● ● ● ● ●        ●  ● ●  ●
● ● ● ● ●         ●  ●  ● ●
● ● ● ● ●        ● ●  ●  ●
● ● ● ● ●         ●  ● ●  ●
(ordered)         (disordered)
```

**Why are they interesting?**
- Exceptional strength (up to 2× steel) with high elasticity
- Superior corrosion and wear resistance
- Unique magnetic properties (soft magnets, transformers)
- Applications: sports equipment, surgical instruments, electronics casings

**The key question:** What determines whether an alloy can form a glass? This is the **glass-forming ability (GFA)** problem.

### Why GNNs for Metallic Glasses?

- Local atomic environment determines properties — perfect for graph representations
- GNNs can capture short/medium-range order even without long-range periodicity
- Composition can itself be represented as a graph (elements as nodes)

### This Notebook

We take a two-part approach:
1. **Part A:** Predict glass-forming ability from alloy *composition* using a GNN
2. **Part B:** Use pretrained universal potentials (M3GNet) on metallic alloy *structures*

This complements Notebook 01 (crystal GNNs) by tackling the **amorphous** side of materials science.